# S14 · Revisión técnica final y hardening
**Escudo Andino · anticipación de heladas en Juliaca** · Big Data UPeU 2026-2 · Equipo 10 BIG IMPACT

*Hardening* = quitar fragilidad a un sistema ya construido, sin agregar funcionalidad. El ciclo de la sesión:
**revisar** la configuración con una lista → **corregir** los fallos → **repetir la ejecución integrada de S13** para demostrar que nada se rompió → **registrar los riesgos que se aceptan**, con su justificación.

> **Aviso de uso:** cambiar un servicio en un `compose.yml` y aplicar `up -d` **recrea su contenedor**. Aplica las correcciones de 3.3 cuando no tengas notebooks o consultas corriendo (recrear `pyspark` cierra los kernels abiertos; recrear Kafka vacía sus topics porque no tiene volumen).

## 3.1 Punto de partida: ejecutar S13 antes de cambiar nada
**Producto del paso:** un manifiesto de S13 *anterior* a los cambios, como punto de comparación. Si S13 no pasa antes, no puedes saber si lo que falle después lo causaste tú.

Con el stack arriba y el productor publicando (S13, 3.1), desde la raíz del repositorio en PowerShell:

```powershell
docker exec lambda26-pyspark jupyter nbconvert --to notebook --execute --inplace /opt/u3/s13-integracion-dataops/13_integracion_dataops.ipynb --ExecutePreprocessor.timeout=900
```

(O abre `13_integracion_dataops.ipynb` en Jupyter y ejecútalo completo.) La celda siguiente lista los manifiestos que existen:

In [ ]:
import glob, json, os, time, urllib.request
import pandas as pd
from datetime import datetime
from IPython.display import Markdown, display

S13_ARTIFACTS = "/opt/u3/s13-integracion-dataops/artifacts"
manifiestos = sorted(glob.glob(S13_ARTIFACTS + "/manifiesto_*.json"))
print("Manifiestos de S13 encontrados:", len(manifiestos))
for m in manifiestos:
    print(" -", os.path.basename(m))
assert manifiestos, "No hay manifiesto: ejecuta primero 13_integracion_dataops.ipynb (S13)"

## 3.2 Revisar la configuración con la lista de comprobación
**Producto del paso:** la lista de comprobación completada con el resultado real de cada punto.

| Punto | Pregunta |
|---|---|
| Versión de las imágenes | ¿Cada imagen tiene versión exacta, no `latest`? |
| Reinicio | ¿Qué hace el contenedor si el proceso cae? (`restart:`) |
| Salud | ¿Hay forma automática de saber si el servicio está **listo**? (`healthcheck:`) |
| Orden de arranque | ¿Se espera a que la dependencia esté **lista**, no solo arrancada? (`depends_on` + `service_healthy`) |
| Secretos | ¿Hay usuarios, claves o tokens escritos en el archivo versionado? |
| Superficie expuesta | ¿Qué puertos se publican y por qué? |

El script de revisión **solo lee** los `compose.yml` (kafka, obs, pyspark y `uso-heladas`) y no modifica nada. Ejecútalo en el **host**, desde la raíz del repositorio, en PowerShell:

```powershell
python pyspark/sesiones/u3/s14-revision-hardening/revision_config.py
```

Y lee su resultado aquí (la carpeta `artifacts/` es la misma que ve el contenedor):

In [ ]:
def cargar_revision(ruta):
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

ARTIFACTS = "./artifacts"
historial = sorted(glob.glob(ARTIFACTS + "/revision_config_*.json"))
assert historial, "Falta el resultado: ejecuta revision_config.py en el host (ver la celda de arriba)"
antes_cfg = cargar_revision(historial[0])          # primera revision = antes de corregir
filas = []
for archivo, lista in antes_cfg["archivos"].items():
    for r in lista:
        filas.append({"archivo": archivo, "servicio": r["servicio"], "imagen": r["imagen"][0], "reinicio": r["reinicio"][0],
                      "salud": r["salud"][0], "orden": r["orden"][0], "secretos": r["secretos"][0], "puertos": r["puertos"][1]})
display(pd.DataFrame(filas))
print("Puntos que FALLAN en la primera revision:", len(antes_cfg["fallas"]))
for f in antes_cfg["fallas"]:
    print(f"  - {f['archivo']} :: {f['servicio']} :: {f['punto']} -> {f['detalle']}")

**Error frecuente:** dar por correcto un punto porque «funciona». La falta de salud no se ve mientras todo está arriba; aparece cuando algo tarda en estar listo, justo la situación que la lista existe para cubrir.

## 3.3 Agregar comprobaciones de salud y esperas por dependencia
**Producto del paso:** los servicios que fallaron en 3.2 con su `healthcheck` y su `restart`.

Kafka y `obs/` ya traen `healthcheck` y `depends_on: service_healthy` (la revisión de 3.2 los marca como `CUMPLE`). Lo que queda son los servicios de **tu proyecto**: `uso-heladas` (sin `restart` ni `healthcheck`) y `pyspark` (sin `healthcheck`).

**Antes de escribirlos, pruébalos a mano dentro del contenedor ya corriendo** (un comando que falla siempre dejaría el servicio `unhealthy` para siempre; el código de salida debe ser `0`):

```powershell
docker exec lambda26-uso-heladas python -c "import socket; socket.create_connection(('kafka', 9092), 5).close()"
docker exec lambda26-pyspark python -c "import socket; socket.create_connection(('localhost', 8888), 5).close()"
```

En `uso-heladas/compose.yml`, agrega `restart` y `healthcheck` al servicio:

```yaml
  uso-heladas:
    # ... (sin cambios)
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import socket; socket.create_connection(('kafka', 9092), 5).close()"]
      interval: 15s
      timeout: 10s
      retries: 5
      start_period: 10s
```

En `pyspark/compose.yml`, agrega el `healthcheck` al servicio `pyspark`:

```yaml
  pyspark:
    # ... (sin cambios)
    healthcheck:
      test: ["CMD", "python", "-c", "import socket; socket.create_connection(('localhost', 8888), 5).close()"]
      interval: 15s
      timeout: 10s
      retries: 10
      start_period: 30s
```

Valida la sintaxis y aplica los cambios (recrea los contenedores: ver el aviso de arriba):

```powershell
docker compose -f uso-heladas/compose.yml config -q
docker compose -f pyspark/compose.yml -f pyspark/compose.kafka.yml config -q
docker compose -f pyspark/compose.yml -f pyspark/compose.kafka.yml up -d --wait
docker compose -f uso-heladas/compose.yml up -d --wait
docker ps --filter name=lambda26 --format "{{.Names}}`t{{.Status}}"
docker exec -d lambda26-uso-heladas python producer_clima.py
```

El `docker exec -d` final es necesario: el contenedor arranca en `sleep infinity` y el productor (y el *consumer*, si lo usas) se lanzan a mano. `depends_on` no existe entre proyectos de Compose distintos; por eso el orden de arranque se garantiza levantando primero Kafka con `up -d --wait`.

**Error frecuente:** dejar `start_period` en cero. Jupyter y Kafka tardan en aceptar conexiones y, sin periodo de gracia, esos segundos cuentan como fallos del `healthcheck`.

## 3.4 Credenciales y secretos fuera del código
**Producto del paso:** ninguna credencial escrita en un archivo versionado.

1. Grafana ya lee su usuario y clave de `${GRAFANA_ADMIN_USER:-admin}` / `${GRAFANA_ADMIN_PASSWORD:-admin}` (3.2: `secretos = CUMPLE`). Compruébalo con la configuración ya interpolada:

```powershell
$env:GRAFANA_ADMIN_PASSWORD = "otra-clave"
docker compose -f obs/compose.yml config | Select-String GF_SECURITY
Remove-Item Env:GRAFANA_ADMIN_PASSWORD
```

Debe mostrar `GF_SECURITY_ADMIN_PASSWORD: otra-clave` y `GF_SECURITY_ADMIN_USER: admin` (el valor por defecto de la variable no definida). Límite: Grafana guarda el administrador en su volumen la **primera** vez; cambiar la variable después no cambia la clave existente.

2. La clave de **WeatherAPI** vive en `pyspark/sesiones/u1/.env`, que no se versiona. Confírmalo:

```powershell
git check-ignore -v pyspark/sesiones/u1/.env
```

Debe responder con la regla del `.gitignore` que lo ignora. Si no responde nada, el archivo se subiría con el siguiente `commit`.

## 3.5 Prueba de no regresión: repetir la ejecución integrada
**Producto del paso:** S13 ejecutado de nuevo sobre la configuración corregida, comparado con el punto de partida de 3.1.

Con 3.3 y 3.4 aplicados, ejecuta otra vez `13_integracion_dataops.ipynb` completo (mismo comando de 3.1). Luego ejecuta esta celda: compara el primer manifiesto (*antes*) con el último (*después*).

In [ ]:
manifiestos = sorted(glob.glob(S13_ARTIFACTS + "/manifiesto_*.json"))
assert len(manifiestos) >= 2, "Necesitas dos ejecuciones de S13 (antes y despues de los cambios). Ejecuta S13 de nuevo."
a = json.load(open(manifiestos[0], encoding="utf-8"))
d = json.load(open(manifiestos[-1], encoding="utf-8"))
print("ANTES  :", a["runId"], "\nDESPUES:", d["runId"], "\n")


def todas_ok(m):
    return all(m["compuertas"].values())


def v(m, *ruta):
    for k in ruta:
        m = m.get(k) if isinstance(m, dict) else None
    return m


comparacion = [
    ("Compuertas previas (todas OK)", todas_ok(a), todas_ok(d)),
    ("Horas del Gold (U1)", v(a, "batch", "gold", "filas"), v(d, "batch", "gold", "filas")),
    ("% de horas con helada", round(v(a, "batch", "gold", "pctHelada"), 3), round(v(d, "batch", "gold", "pctHelada"), 3)),
    ("Latencia p50 (ms)", v(a, "batch", "eventosS9", "latenciaP50Ms"), v(d, "batch", "eventosS9", "latenciaP50Ms")),
    ("Latencia p95 (ms)", v(a, "batch", "eventosS9", "latenciaP95Ms"), v(d, "batch", "eventosS9", "latenciaP95Ms")),
    ("Predicciones del stream", v(a, "streaming", "filas"), v(d, "streaming", "filas")),
    ("MAE del modelo (C)", None if v(a, "streaming", "maeModelo") is None else round(v(a, "streaming", "maeModelo"), 3),
     None if v(d, "streaming", "maeModelo") is None else round(v(d, "streaming", "maeModelo"), 3)),
    ("Metricas confirmadas en Prometheus", v(a, "observabilidad", "todasEnPrometheus"), v(d, "observabilidad", "todasEnPrometheus")),
    ("Duracion (s)", v(a, "duracionTotalS"), v(d, "duracionTotalS")),
]
display(pd.DataFrame(comparacion, columns=["Comprobacion", "Antes (S13)", "Despues (S14)"]))

# Regresion = algo que funcionaba y dejo de funcionar. Que el MAE o el numero de lecturas cambien NO lo es:
# el topic crece con lecturas nuevas (fuente en vivo).
criterios = {
    "Las compuertas previas siguen pasando": todas_ok(d),
    "El Gold no cambio (mismas horas)": v(a, "batch", "gold", "filas") == v(d, "batch", "gold", "filas"),
    "El stream sigue procesando filas (> 0)": (v(d, "streaming", "filas") or 0) > 0,
    "Prometheus devuelve los valores recien publicados": bool(v(d, "observabilidad", "todasEnPrometheus")),
}
for nombre, ok in criterios.items():
    print(("OK    " if ok else "FALLA ") + nombre)
regresion = not all(criterios.values())
resultado_no_regresion = {"antes": a["runId"], "despues": d["runId"], "criterios": criterios, "hayRegresion": regresion}
assert not regresion, "HAY REGRESION: corrige el ultimo cambio y repite S13"
print("\nSin regresion: el sistema integrado sigue funcionando tras los cambios.")

**Error frecuente:** dar por pasada la prueba porque «no hubo errores». Un resultado sin errores solo demuestra que nada falló de forma ruidosa; la tabla compara resultados, que es lo que detecta un cambio silencioso.

## 3.6 Riesgos aceptados
**Producto del paso:** la tabla de riesgos que **no** se corrigen, cada uno con su justificación y lo que cambiaría en producción. Un riesgo sin esa constancia no está aceptado: está olvidado.

In [ ]:
riesgos = [
    ("Kafka sin volumen: recrear el contenedor vacia el topic `heladas-eventos`.",
     "Laboratorio reproducible desde cero; el historico completo vive en el Gold de U1 (146 832 horas) y el productor re-llena el topic.",
     "Volumen persistente para los datos del broker y replicacion entre mas de un broker."),
    ("Kafka en PLAINTEXT, sin autenticacion.",
     "La red `lambda26-kafka-net` es local; TLS y SASL complicarian cada ejemplo sin aportar al tema de la unidad.",
     "Cifrado y autenticacion en cada listener."),
    ("Grafana con `admin`/`admin` por defecto y Jupyter con token `sintoken`.",
     "El sistema debe arrancar sin configuracion previa; son herramientas locales del estudiante.",
     "Clave propia desde el primer arranque (variable no versionada) y Jupyter detras de acceso autenticado."),
    ("Todos los puertos (49092, 49090, 43000, 4488, ...) publicados al host.",
     "Se necesita llegar a cada herramienta desde el navegador del laboratorio.",
     "Publicar solo el punto de entrada; el resto dentro de la red."),
    ("Dependencia de una clave de WeatherAPI de prueba (plan Business, vence ~27-nov-2026).",
     "El Gold ya esta descargado y verificado; el modelo y el stream no necesitan la clave para evaluarse. Solo el productor en vivo la usa.",
     "Plan de pago vigente o proveedor alterno, y clave en un gestor de secretos."),
    ("Modelo entrenado una sola vez (2010-2026), sin reentrenamiento automatico ni vigilancia de deriva.",
     "Alcance del curso: el modelo se evalua y se compara con persistencia, y la ventaja real se midio hasta 3-4 h.",
     "Reentrenar con una frecuencia fija y alertar cuando el MAE en produccion supere al de persistencia."),
    ("Lecturas horarias: el historico de WeatherAPI no tiene resolucion de 15 min con esta clave.",
     "Se entrena y evalua a frecuencia horaria; las lecturas de 15 min se guardan pero el modelo solo usa pares a 1 h exacta.",
     "Fuente con resolucion sub-horaria para el historico."),
]
tabla = "| Riesgo | Por que se acepta aqui | Que cambiaria en produccion |\n|---|---|---|\n"
for r in riesgos:
    tabla += "| " + " | ".join(r) + " |\n"
display(Markdown(tabla))
os.makedirs("./artifacts", exist_ok=True)
with open("./artifacts/riesgos_aceptados.md", "w", encoding="utf-8") as f:
    f.write("# Riesgos aceptados - Escudo Andino (heladas en Juliaca)\n\n" + tabla)
print("Guardado en ./artifacts/riesgos_aceptados.md")

## 3.7 Métricas y evidencias para la demostración
**Producto del paso:** las cifras reales del sistema que sostendrás en la demo, **cada una con su origen**. Una cifra sin origen es una afirmación.

In [ ]:
manifiestos = sorted(glob.glob(S13_ARTIFACTS + "/manifiesto_*.json"))
ult = json.load(open(manifiestos[-1], encoding="utf-8"))
b, e = ult["batch"], ult["streaming"]


def prom(nombre):
    try:
        res = json.load(urllib.request.urlopen("http://prometheus:9090/api/v1/query?query=" + nombre, timeout=5))["data"]["result"]
        return res[0]["value"][1] if res else "sin dato"
    except Exception:
        return "Prometheus no responde"


metricas = [
    ("Horas reales en el Gold (U1)", b["gold"]["filas"], "Manifiesto, celda 3.4 de S13", prom("lambda26_int_gold_horas_total")),
    ("% de horas con helada 2010-2026", round(b["gold"]["pctHelada"], 3), "Manifiesto, celda 3.4 de S13", prom("lambda26_int_gold_helada_pct")),
    ("Eventos en el Parquet de S9", b["eventosS9"]["filas"], "Manifiesto, celda 3.4 de S13", prom("lambda26_int_batch_eventos_total")),
    ("Ratio de eventos validos", round(b["eventosS9"]["ratioValidos"], 4), "Manifiesto", prom("lambda26_int_batch_validos_ratio")),
    ("Latencia tipica (p50, ms)", b["eventosS9"]["latenciaP50Ms"], "Manifiesto", prom("lambda26_int_batch_latencia_p50_ms")),
    ("Latencia de la cola (p95, ms)", b["eventosS9"]["latenciaP95Ms"], "Manifiesto", prom("lambda26_int_batch_latencia_p95_ms")),
    ("Predicciones del stream", e["filas"], "Manifiesto, celda 3.5 de S13", prom("lambda26_int_stream_filas_total")),
    ("MAE del modelo en produccion (C)", None if e["maeModelo"] is None else round(e["maeModelo"], 3),
     "Manifiesto, celda 3.6 de S13", prom("lambda26_int_stream_mae")),
    ("MAE de persistencia (C)", None if e["maePersistencia"] is None else round(e["maePersistencia"], 3),
     "Manifiesto, celda 3.6 de S13", prom("lambda26_int_stream_mae_persistencia")),
    ("Duracion de la ejecucion integrada (s)", ult["duracionTotalS"], "Manifiesto", "-"),
]
display(pd.DataFrame(metricas, columns=["Metrica", "Valor (manifiesto)", "Origen", "En Prometheus ahora"]))
print("Manifiesto usado:", os.path.basename(manifiestos[-1]))

Cifras de unidades anteriores que también defiendes (cada una sale de su notebook): U1 Random Forest RMSE 1,077 °C · R² 0,801 · recall 0,875 con margen de 0,75 °C; U2 a 1 h: RMSE 0,971 °C frente a 1,499 °C de persistencia (−35,2 %); ventaja real hasta 3-4 h; costo estimado US$ 91,16 al mes.

## 3.8 Guion de demostración
**Producto del paso:** el guion con tiempos, qué muestras, qué debe verse y un **plan alternativo** por paso. Ensáyalo completo con cronómetro y **verifica que el productor esté publicando antes de empezar**.

In [ ]:
guion = [
    ("0-1", "Entorno sano.", "`docker ps --filter name=lambda26`", "Los servicios con `(healthy)`.", "Captura con fecha y hora."),
    ("1-3", "Compuertas previas.", "Celda 3.3 de S13.", "Ocho `OK`, incluido el productor activo.", "Captura de una corrida anterior."),
    ("3-5", "Batch del Gold y latencia.", "Celda 3.4 de S13.", "146 832 horas, % de helada y p50 frente a p95.", "Salida guardada del notebook."),
    ("5-7", "Streaming con el modelo.", "Celdas 3.5-3.6 de S13.", "Micro-lotes con filas y MAE del modelo frente a persistencia.", "Manifiesto de una corrida anterior."),
    ("7-9", "Observabilidad.", "Prometheus (`:49090`) y Grafana (`:43000`), tablero *S13 - Integracion Heladas Juliaca*.", "Metricas `lambda26_int_*` y la antiguedad de la ultima ejecucion.", "Captura del tablero."),
    ("9-10", "Registro de la ejecucion.", "`artifacts/manifiesto_<runId>.json`.", "Compuertas, KPI, bitacora y duracion.", "El archivo abierto."),
]
tabla = "| Min. | Que muestras | Comando o URL | Que debe verse | Plan alternativo |\n|---:|---|---|---|---|\n"
for g in guion:
    tabla += "| " + " | ".join(g) + " |\n"
display(Markdown(tabla))

Antes de empezar la demo, esta celda comprueba que todo está listo (solo lecturas):

In [ ]:
def listo(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False


manifiestos = sorted(glob.glob(S13_ARTIFACTS + "/manifiesto_*.json"))
edad_h = (time.time() - os.path.getmtime(manifiestos[-1])) / 3600 if manifiestos else None
listos = {
    "Prometheus listo": listo("http://prometheus:9090/-/ready"),
    "Pushgateway listo": listo("http://pushgateway:9091/-/ready"),
    "Grafana saludable": listo("http://grafana:3000/api/health"),
    "Existe un manifiesto de S13": bool(manifiestos),
    "Manifiesto de menos de 24 h": edad_h is not None and edad_h < 24,
}
for n, ok in listos.items():
    print(("OK    " if ok else "FALLA ") + n)
print("\nRecuerda: confirma que el productor publica (offset creciente) antes de empezar.")

## 3.9 Explicación técnica del sistema
**Producto del paso:** las decisiones del sistema, por qué se tomaron y qué cuestan. Es la defensa técnica en una tabla.

In [ ]:
decisiones = [
    ("Arquitectura Lambda: batch para aprender y streaming para reaccionar.",
     "El Gold de 16 anos entrena el modelo (sin prisa) y el stream aplica ese modelo a lecturas nuevas con baja latencia.",
     "Dos caminos de codigo que mantener; se mitiga reutilizando el mismo `preparar()` en lote y streaming."),
    ("Publicar los resultados de una ejecucion en Pushgateway.",
     "Un job corto no puede ser raspado por Prometheus: el mecanismo correcto es empujar.",
     "Prometheus no sabe si el dato es reciente; de ahi el panel de antiguedad de la ultima ejecucion."),
    ("Percentiles de latencia (p50/p95) en vez del promedio.",
     "El promedio de una distribucion con cola larga no representa a ningun grupo.",
     "Mas metricas y paneles que explicar."),
    ("Compuertas que detienen la ejecucion.",
     "Un resultado vacio o absurdo se detecta donde nace, con el nombre de lo que falta.",
     "Una compuerta mal calibrada detiene un pipeline sano: por eso el p95 es advertencia y el p50 es compuerta."),
    ("Streaming acotado con `availableNow` y no una consulta infinita.",
     "Una ejecucion integrada tiene que terminar para poder comprobarse; el productor publica solo cada 15 min.",
     "Evalua el modelo sobre lo publicado hasta ahora, no sobre el futuro."),
    ("Comparar el modelo contra persistencia y no solo reportar el RMSE.",
     "Sin una referencia simple no se sabe si el modelo aporta; la ventaja real se midio hasta 3-4 h.",
     "Mas de 4 h el modelo no supera de forma fiable a persistencia: se declara como limite."),
]
tabla = "| Decision | Por que | Que se pierde |\n|---|---|---|\n"
for dd in decisiones:
    tabla += "| " + " | ".join(dd) + " |\n"
display(Markdown(tabla))

## 3.10 Registrar los hallazgos en la bitácora
**Producto del paso:** una fila por hallazgo. Se arma sola desde la revisión de 3.2 y la prueba de 3.5, y se guarda el consolidado de la revisión técnica.

Para que la celda muestre el **después**, ejecuta de nuevo `revision_config.py` en el host (3.2) una vez aplicadas las correcciones: la celda compara la primera revisión (*antes*) con la última (*después*).

In [ ]:
historial = sorted(glob.glob("./artifacts/revision_config_*.json"))
antes_cfg = cargar_revision(historial[0])
despues_cfg = cargar_revision(historial[-1])
corregidos = {(f["archivo"], f["servicio"], f["punto"]) for f in antes_cfg["fallas"]} - \
             {(f["archivo"], f["servicio"], f["punto"]) for f in despues_cfg["fallas"]}
pendientes = despues_cfg["fallas"]

CAUSA = {"salud": "Los compose se escribieron con la configuracion minima de clase, sin `healthcheck`.",
         "reinicio": "El servicio no declaraba `restart`: si el proceso cae, no vuelve solo.",
         "orden": "`depends_on` esperaba solo a que el contenedor arrancara.",
         "secretos": "Valor escrito directamente en el archivo versionado.",
         "imagen": "Imagen sin version exacta."}
CORR = {"salud": "`healthcheck` probado a mano y aplicado (3.3).",
        "reinicio": "`restart: unless-stopped` (3.3).",
        "orden": "`depends_on` con `condition: service_healthy`.",
        "secretos": "Variable `${...}` leida de fuera del archivo (3.4).",
        "imagen": "Version exacta de la imagen."}

filas = []
for f in antes_cfg["fallas"]:
    clave = (f["archivo"], f["servicio"], f["punto"])
    estado = "corregido" if clave in corregidos else "PENDIENTE"
    filas.append((f"`{f['archivo']}` :: {f['servicio']}: {f['detalle']}.", CAUSA.get(f["punto"], "-"), CORR.get(f["punto"], "-"),
                  f"Segunda ejecucion de `revision_config.py`: {estado}."))
if not filas:
    filas.append(("La primera revision no encontro fallas de configuracion.", "-", "-", "`revision_config.py` sin puntos FALLA."))
try:
    nr = resultado_no_regresion
    filas.append(("Los cambios de configuracion podian romper la ejecucion integrada.", "Un cambio en un compose recrea contenedores.",
                  "Prueba de no regresion: se repite S13 tras los cambios (3.5).",
                  f"Manifiestos {nr['antes']} vs {nr['despues']}: " + ("sin regresion." if not nr["hayRegresion"] else "HAY REGRESION.")))
except NameError:
    print("AVISO: ejecuta la celda de 3.5 para incluir la prueba de no regresion.")

tabla = "| # | Hallazgo | Causa | Correccion | Prueba de correccion |\n|---|---|---|---|---|\n"
for i, fila in enumerate(filas, 1):
    tabla += f"| {i} | " + " | ".join(fila) + " |\n"
display(Markdown(tabla))
print("Fallas que siguen pendientes:", len(pendientes))

final = {"fecha": datetime.now().isoformat(timespec="seconds"),
         "revisionAntes": historial[0], "revisionDespues": historial[-1],
         "fallasAntes": antes_cfg["fallas"], "fallasDespues": pendientes,
         "noRegresion": globals().get("resultado_no_regresion"),
         "riesgosAceptados": len(globals().get("riesgos", [])),
         "bitacora": [dict(zip(("hallazgo", "causa", "correccion", "prueba"), f)) for f in filas]}
with open("./artifacts/revision_tecnica_final.json", "w", encoding="utf-8") as f:
    json.dump(final, f, indent=2, ensure_ascii=False)
print("Consolidado guardado en ./artifacts/revision_tecnica_final.json")

### Evidencia de aprendizaje
- Lista de comprobación completada con el resultado real de cada punto (3.2).
- Comprobaciones de salud probadas a mano, aplicadas y con `docker ps` en `(healthy)` (3.3).
- Credenciales verificadas con `docker compose config` y `git check-ignore` (3.4).
- Prueba de no regresión con la tabla antes/después (3.5) · riesgos aceptados con justificación (3.6).
- Métricas con su origen, guion de demostración y explicación técnica (3.7-3.9) · bitácora (3.10).